# M1 on Kaggle: first real model (~6M params, TinyStories)

Settings → **Accelerator: GPU (T4 ×2 or P100)**, **Internet: on** (needs a phone-verified Kaggle account).

This notebook:
1. installs the repo, 2. smoke-tests the GPU code path, 3. prepares TinyStories,
4. trains with a **deliberate interruption**, 5. resumes, 6. writes the report.

Optional secrets (Add-ons → Secrets): `HF_TOKEN` (upload checkpoints to the Hugging Face Hub),
`GITHUB_TOKEN` (only if the repo is private). Never paste tokens into cells.

In [ ]:
import os, subprocess
BRANCH = "claude/zealous-lamport-u4ady6"
url = "https://github.com/thedeepakreddy/LLM.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{tok}@github.com/thedeepakreddy/LLM.git"
except Exception:
    pass  # public repo or no secret set
if not os.path.exists("LLM"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "LLM"], check=True)
%cd LLM
!pip install -q -e .

## 1. GPU check and smoke test

In [ ]:
import torch
from llm.train.loop import resolve_device, resolve_precision
dev = resolve_device("auto")
print(torch.__version__, dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "")
print("precision:", resolve_precision("auto", dev))   # T4/P100 -> fp16 + GradScaler

In [ ]:
# Tiny overfit run on the GPU: proves the fp16/bf16 path works before the real run.
!llm prepare-data configs/m1-overfit.yaml
!llm train-tokenizer configs/m1-overfit.yaml
!llm tokenize configs/m1-overfit.yaml
!llm train configs/m1-overfit.yaml --set train.precision=auto --set train.max_steps=300

## 2. Prepare TinyStories (download, clean, tokenizer, shards)
The tokenizer and tokenization steps are pure Python and CPU-bound; expect tens of minutes.

In [ ]:
CFG = "configs/m1-tinystories.yaml"
!llm prepare-data {CFG}
!cat data/tinystories-v1/data_report.md
!llm train-tokenizer {CFG}
!llm tokenize {CFG}

## 3. Train, with a deliberate interruption at step 500

In [ ]:
!llm train {CFG} --stop-at-step 500

In [ ]:
import glob
RUN = sorted(glob.glob("runs/m1-tinystories-*"))[-1]
print(RUN)
!tail -3 {RUN}/metrics.jsonl

## 4. Resume to the end
If the session dies, re-run the install cell, set `RUN`, and run this cell again: it continues from the last valid checkpoint.

In [ ]:
!llm resume {RUN}

## 5. Generate, report

In [ ]:
!llm generate {RUN} --prompt "Once upon a time, there was a little dog named" --seed 1
!llm generate {RUN} --prompt "Lily wanted to go to the park, but" --seed 1

In [ ]:
# Write honest limitations before sharing, then build the report.
open(f"{RUN}/limitations.md", "w").write(
    "# Known limitations\n\n"
    "- ~6M parameters trained on synthetic TinyStories only: simple children's-story English.\n"
    "- No instruction tuning; it continues text rather than answering questions.\n"
    "- Single run, single seed; no ablations yet.\n")
!llm report {RUN}
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN}/report.md").read().replace("](charts/", f"]({RUN}/charts/")))

## 6. (Optional) Upload the run to the Hugging Face Hub

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
repo = api.whoami()["name"] + "/llm-m1-tinystories"
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=RUN, repo_id=repo, path_in_repo=RUN.split("/")[-1])
print("uploaded to", repo)